# סיווג משלחי יד מתוך תיאורי עבודה
**שולרופר אור | מטלת Data Scientist | הלשכה המרכזית לסטטיסטיקה**

מחברת זו מממשת תהליך מלא על הנתונים המדומים שסופקו: קריאה, בדיקות איכות, בניית מודל, הערכה והפקת הצעות לבדיקה אנושית. משתנה המטרה הוא `SemelMishlachSofi`. אין לראות בציונים הוכחה לביצועים על נתוני אמת.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/main/project_notebooks/cbs_occupation_classification/Assignment.ipynb)

**הפעלה ב־Codespaces:** פותחים את המחברת מתוך המאגר ומפעילים Run all. הקובץ נמצא בתיקייה `Datasets/cbs_occupation_survey_sample`.

**הפעלה ב־Colab או במחברת המצורפת למייל:** מעלים את `DUMMY_DATASET.xlsx` ללשונית Files של סביבת הריצה, ואז מפעילים Runtime > Run all. אפשר גם להשתמש בעותק קיים של המאגר או להגדיר DATASETS_ROOT. המחברת קוראת נתונים קיימים בלבד, בהתאם למבנה המאגר. כשאין קובץ היא מדווחת Dataset not found ומדלגת על חישובים תלויים. תוצאה זו אינה הרצת מודל מוצלחת. אין צורך ב־GPU או במפתח API.

**קישור לקוד ב־GitHub:** https://github.com/shulruferor-sudo/Advance-Data-Science/blob/main/project_notebooks/cbs_occupation_classification/Assignment.ipynb

**מצב בדיקה:** תאי Python הורצו מקומית, ואותו קוד עבר גם הרצה מלאה בגרעין Jupyter חדש ב־GitHub Actions, עם 14 בדיקות תקינות ואימות טביעת קובץ המקור. [תיעוד ההרצה](https://github.com/shulruferor-sudo/Advance-Data-Science/actions/runs/41051182364). קישור Colab נפתח ומציג את המחברת. הרצה בשירות Colab עצמו לא אומתה, מפני שסביבת הדפדפן דורשת כניסה לחשבון Google. הפלטים השמורים במחברת נוצרו בהרצה המקומית.


## 1. הגדרת הבעיה והחלטות לפני ההערכה
המטרה העסקית היא לעזור לסמלנים לבחור קוד משלח יד מתוך תיאור העבודה, לצמצם עבודה חוזרת ולהפנות מקרים חסרים או עמומים לבדיקה. הצלחה עתידית תימדד גם באיכות הסימול ובזמן הטיפול.

המודל הראשי נקבע מראש: ייצוג TF-IDF של מילים ורצפי תווים ומסווג LinearSVC עם C=1. ייצוג זה מודד דפוסי שימוש בטקסט. הוא אינו מבטיח הבנת משמעות. נשווה למודל שמנבא תמיד את המחלקה הנפוצה ולגרסאות מילים בלבד, תווים בלבד ותיאור תפקיד בלבד. לא נשנה פרמטרים לפי ציוני ההערכה.

הקלט הראשי כולל את תיאור העבודה, הפעולות, התפקיד והקשר הארגוני. מזהה הרשומה, קוד משלח היד וקוד הענף הכלכלי אינם קלט. גם גיל, יישוב, השכלה ותאריך תחילת תפקיד אינם קלט במודל ההדגמה. זמינות קוד הענף במועד החיזוי לא הוגדרה. ID הוא מספר רץ לפי Metadata, ולא מזהה אדם. תאריך תחילת תפקיד אינו זמן שליחת הסקר.

נשמור תיאורי תפקיד זהים באותה קבוצת הערכה, באמצעות שלושת השדות EzoAvoda, TeurPeula ו־TeurTafkid לאחר נרמול בסיסי. זו הגנה מפני חזרת טקסט, ולא זיהוי מוכח של אותו נסקר. ייתכנו ניסוחים קרובים בקבוצות שונות.

## סביבת הריצה
התא הבא בודק תלויות, מגדיר נתיבים ומדווח גרסאות.


In [1]:
import sys, subprocess, importlib.util, json, re, unicodedata, hashlib, time
from pathlib import Path
import os
from contextlib import contextmanager
from datetime import datetime, timezone

# Install only missing dependencies. Colab usually already provides these.
for module, package in [("pandas", "pandas==2.2.3"), ("numpy", "numpy==2.3.5"),
                        ("sklearn", "scikit-learn==1.8.0"), ("openpyxl", "openpyxl==3.1.5")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
import numpy as np
import pandas as pd
import sklearn, openpyxl
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42
REPOSITORY_ROOT = Path(os.environ.get("ADVANCED_DATA_SCIENCE_ROOT", Path.cwd())).expanduser().resolve()
for candidate in (REPOSITORY_ROOT, *REPOSITORY_ROOT.parents):
    if (candidate / "Datasets").is_dir() and (candidate / "project_notebooks").is_dir():
        REPOSITORY_ROOT = candidate
        break
DATASETS_ROOT = Path(os.environ.get("DATASETS_ROOT", REPOSITORY_ROOT / "Datasets")).expanduser()
PROJECT_DIRECTORY = REPOSITORY_ROOT / "project_notebooks" / "cbs_occupation_classification"
if not PROJECT_DIRECTORY.is_dir():
    PROJECT_DIRECTORY = Path.cwd()
OUTPUT = PROJECT_DIRECTORY / "results"
OUTPUT.mkdir(exist_ok=True)
VERSIONS = {"python": sys.version.split()[0], "pandas": pd.__version__,
            "numpy": np.__version__, "scikit-learn": sklearn.__version__,
            "openpyxl": openpyxl.__version__}
print("Environment:", json.dumps(VERSIONS))
DECLARED_VERSIONS={"pandas":"2.2.3","numpy":"2.3.5","scikit-learn":"1.8.0","openpyxl":"3.1.5"}
for package,version in DECLARED_VERSIONS.items():
    if VERSIONS[package]!=version:
        print(f"Version difference: {package}={VERSIONS[package]}, reference={version}. Existing installation retained.")

@contextmanager
def report_failure(stage):
    try:
        yield
    except Exception as error:
        print(f"FAILED: {stage}: {type(error).__name__}: {error}")
        print("Inspect the stated input, dependency or operation before rerunning dependent cells.")
        raise


Environment: {"python": "3.12.14", "pandas": "2.2.3", "numpy": "2.3.5", "scikit-learn": "1.8.0", "openpyxl": "3.1.5"}


## 2. קריאת המקור ובדיקת מבנה
נקרא את 18 השדות המוגדרים בגיליון Survey Dataset ואת ארבע עמודות ההסבר בגיליון Metadata. בקובץ יש גם נוסחה ב־J202 שאינה רשומת סקר, והעתקות של שם שדה בעמודות רחוקות בגיליון ההסבר. אין לשנות את המקור. שורות ללא ID תקין נשמרות בנפרד לצורך תיעוד ולא נכנסות לאימון.


In [2]:
with report_failure("Read supplied survey"):
    DATA_PATH = DATASETS_ROOT / "cbs_occupation_survey_sample" / "DUMMY_DATASET.xlsx"
    if not DATA_PATH.is_file() and Path("DUMMY_DATASET.xlsx").is_file():
        DATA_PATH = Path("DUMMY_DATASET.xlsx")

    EXPECTED_COLUMNS = ["ID", "YeshuvAvoda", "ShemAvoda", "SugAvoda", "ShemMachlaka",
        "SugMachlaka", "EzoAvoda", "TeurPeula", "TeurTafkid", "MenahelEtMi", "MaamadAvoda",
        "MakorSachar", "TeudaGvoha", "shnotlimud", "Gil", "TarichHathala", "SemelAnafSofi", "SemelMishlachSofi"]

    def read_survey(path):
        book = pd.ExcelFile(path, engine="openpyxl")
        required = {"Survey Dataset", "Metadata"}
        if not required.issubset(book.sheet_names):
            raise ValueError("Missing required worksheet: " + str(required - set(book.sheet_names)))
        raw = pd.read_excel(book, sheet_name="Survey Dataset", usecols="A:R", dtype=str).fillna("")
        if list(raw.columns) != EXPECTED_COLUMNS:
            raise ValueError("Input schema differs from the supplied assignment workbook.")
        valid = raw["ID"].str.fullmatch(r"[1-9]\d*")
        excluded = raw.loc[~valid].copy()
        data = raw.loc[valid].copy().reset_index(drop=True)
        if data.empty or data["ID"].duplicated().any():
            raise ValueError("No usable records or duplicate IDs. Inspect the input before training.")
        metadata = pd.read_excel(book, sheet_name="Metadata", usecols="A:D", dtype=str).fillna("")
        return data, excluded, metadata

    df = None
    try:
        df, excluded, metadata = read_survey(DATA_PATH)
    except FileNotFoundError:
        print(f"Dataset not found: {DATA_PATH}. Upload the supplied workbook or prepare it in Datasets. Dependent calculations are skipped.")
    if df is not None:
        DATA_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
        print("Dataset SHA-256:", DATA_SHA256)
        print("Survey records:", len(df), "Columns:", len(df.columns), "Excluded nonrecords:", len(excluded))
        print(metadata.to_string(index=False))
        print("Excluded populated cells:", [{k:v for k,v in row.items() if v} for row in excluded.to_dict("records")])


Dataset SHA-256: e1e7f391c0675dd14830eaae6b5cd114edc256f18b956bb48a454b0b1209e1ac
Survey records: 200 Columns: 18 Excluded nonrecords: 1
            משתנה                                                                                                  הסבר   פורמט                                                                                                                                                                                                                                                                                                                             ערכים אפשריים
               ID                                                                                               מספר רץ INTEGER                                                                                                                                                                                                                                                                                      

## 3. נרמול, סימולים חלקיים וחזרות
הנרמול מוגבל ל־Unicode NFKC, אותיות קטנות באנגלית וצמצום רווחים. אין תיקון אוטומטי של מילים, תרגום, השלמת עובדות או שינוי תוויות. הביטוי כנ״ל נשאר כפי שנכתב, מאחר שלא ברור תמיד לאיזה שדה הוא מתייחס.

סימול מלא הוא ארבע ספרות. סימול הכולל X נשמר כרשומה לסיוע ולבדיקה, אך אינו משמש אמת מלאה לאימון או לחישוב דיוק. טענת כ־90% נכונות הסימול הידני מופיעה ברקע המטלה. אין כאן תוויות אמת חיצוניות שמאפשרות לאמת אותה או לזהות את כל הטעויות.


In [3]:
if df is None:
    print("Skipped: dataset not available.")
else:
    with report_failure("Assignment stage 3"):
        TARGET = "SemelMishlachSofi"
        TEXT_FIELDS = ["ShemAvoda", "SugAvoda", "ShemMachlaka", "SugMachlaka", "EzoAvoda", "TeurPeula", "TeurTafkid"]
        CORE_FIELDS = ["EzoAvoda", "TeurPeula", "TeurTafkid"]

        def normalize(value):
            if pd.isna(value):
                return ""
            return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", str(value)).strip().lower())

        def text_for(frame, fields=TEXT_FIELDS):
            # Separate field boundaries. Do not infer the meaning of missing values or 'ditto'.
            return frame[fields].fillna("").apply(lambda row: " \n ".join(normalize(v) for v in row), axis=1)

        def group_for(frame):
            signatures = frame[CORE_FIELDS].apply(lambda row: tuple(normalize(v) for v in row), axis=1)
            # If all task fields are empty, use available organizational text conservatively.
            for i, signature in signatures.items():
                if not any(signature):
                    signatures.at[i] = tuple(normalize(frame.at[i, c]) for c in TEXT_FIELDS)
            return signatures.map(lambda x: hashlib.sha256(json.dumps(x, ensure_ascii=False).encode()).hexdigest())

        labels = df[TARGET].str.strip().str.upper()
        complete_mask = labels.str.fullmatch(r"\d{4}")
        partial_mask = labels.str.fullmatch(r"[0-9X]{4}") & labels.str.contains("X")
        if not (complete_mask | partial_mask).all():
            raise ValueError("Unexpected label format. Inspect and document rather than silently repairing it.")
        complete = df.loc[complete_mask].reset_index(drop=True)
        partial = df.loc[partial_mask].reset_index(drop=True)
        y = complete[TARGET].str.strip().to_numpy()
        groups = group_for(complete).to_numpy()
        classes = np.sort(np.unique(y))
        counts = pd.Series(y).value_counts()
        feature_cols = [c for c in EXPECTED_COLUMNS if c not in {"ID", TARGET}]
        duplicate_sets = []
        for _, block in df.groupby(feature_cols, dropna=False, sort=False):
            if len(block) > 1 and block[TARGET].nunique() > 1:
                duplicate_sets.append({"ids": block.ID.tolist(), "labels": block[TARGET].tolist()})
        quality = {
            "records":len(df), "fields":len(df.columns), "complete_labels":len(complete),
            "partial_labels":len(partial), "complete_classes":len(classes),
            "singleton_classes":int((counts == 1).sum()), "task_groups":int(len(np.unique(groups))),
            "excluded_nonrecords":len(excluded),
            "duplicate_feature_rows":int(df[feature_cols].duplicated(keep=False).sum()),
            "conflicting_exact_feature_sets":duplicate_sets,
            "missing_by_field":{c:int(df[c].eq("").sum()) for c in EXPECTED_COLUMNS},
            "partial_codes":partial[TARGET].value_counts().to_dict()
        }
        print(json.dumps(quality, ensure_ascii=False, indent=2))
        print("Complete-label class counts:")
        print(counts.to_string())


{
  "records": 200,
  "fields": 18,
  "complete_labels": 191,
  "partial_labels": 9,
  "complete_classes": 59,
  "singleton_classes": 24,
  "task_groups": 90,
  "excluded_nonrecords": 1,
  "duplicate_feature_rows": 22,
  "conflicting_exact_feature_sets": [
    {
      "ids": [
        "2",
        "33",
        "65",
        "96"
      ],
      "labels": [
        "3311",
        "3211",
        "3211",
        "3211"
      ]
    }
  ],
  "missing_by_field": {
    "ID": 0,
    "YeshuvAvoda": 0,
    "ShemAvoda": 0,
    "SugAvoda": 25,
    "ShemMachlaka": 64,
    "SugMachlaka": 134,
    "EzoAvoda": 7,
    "TeurPeula": 136,
    "TeurTafkid": 172,
    "MenahelEtMi": 168,
    "MaamadAvoda": 2,
    "MakorSachar": 14,
    "TeudaGvoha": 2,
    "shnotlimud": 2,
    "Gil": 2,
    "TarichHathala": 0,
    "SemelAnafSofi": 0,
    "SemelMishlachSofi": 0
  },
  "partial_codes": {
    "1XXX": 4,
    "XXXX": 4,
    "335X": 1
  }
}
Complete-label class counts:
5223    14
9112    12
2512    10
5120     7

## 4. הערכה עם הפרדה בין קבוצות תיאור
כל אחת מ־191 הרשומות עם סימול מלא תקבל חיזוי ממודל שלא התאמן על הקבוצה שלה. נשתמש בחמישה קפלים של GroupKFold. בכל קפל גם מילון המילים ומשקלי TF-IDF נלמדים רק מנתוני האימון.

**המדדים:** Accuracy הוא שיעור התאמת הקוד המלא. Macro F1 נותן משקל שווה לכל אחת מ־59 המחלקות ומציג גם את הקושי במחלקות נדירות. Top 3 הוא שיעור המקרים שבהם התווית מופיעה בין שלוש ההצעות. בנוסף יוצג שיעור התאמת הספרה הראשונה, שאינו שקול לסימול מלא.

חלק מהקפלים מכילים קודים שאינם קיימים באימון שלהם. אלו מקרים אמיתיים של מגבלת המסווג הסגור, ונשאיר אותם בחישוב הראשי. נציג בנפרד גם דיוק רק כאשר הקוד כבר הופיע באימון. קבוצות הן תיאורי טקסט, לא אנשים. ההערכה אינה ניסוי עתידי או מדידה של נתוני אמת.


In [4]:
if df is None:
    print("Skipped: dataset not available.")
else:
    with report_failure("Assignment stage 4"):
        def make_model(kind="combined"):
            word = TfidfVectorizer(ngram_range=(1,2), token_pattern=r"(?u)\b\w+\b", sublinear_tf=True)
            char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2,5), sublinear_tf=True)
            if kind == "dummy":
                return DummyClassifier(strategy="most_frequent")
            feature = word if kind == "word" else char if kind == "char" else FeatureUnion([("word",word),("char",char)])
            return Pipeline([("features",feature), ("classifier",LinearSVC(C=1.0, dual=True, random_state=SEED, max_iter=20000))])

        def ranks_and_margin(model, text):
            scores = np.asarray(model.decision_function(text))
            if scores.ndim == 1:
                scores = np.column_stack([-scores, scores])
            rank = np.argsort(-scores, axis=1, kind="stable")[:, :min(3, len(model.classes_))]
            ranked = model.classes_[rank]
            ordered = np.sort(scores, axis=1)
            margins = ordered[:,-1] - ordered[:,-2]
            return ranked, margins

        def evaluate(kind, fields=TEXT_FIELDS, split_kind="grouped"):
            text = text_for(complete, fields).to_numpy()
            if split_kind == "grouped":
                splits = GroupKFold(n_splits=5).split(text,y,groups)
            else:
                splits = KFold(n_splits=5,shuffle=True,random_state=SEED).split(text)
            rows, fold_stats = [], []
            for fold,(train_idx,test_idx) in enumerate(splits,1):
                shared_groups = set(groups[train_idx]) & set(groups[test_idx])
                if split_kind == "grouped":
                    assert not shared_groups
                model=make_model(kind)
                started=time.perf_counter()
                if kind == "dummy":
                    model.fit(np.zeros((len(train_idx),1)),y[train_idx])
                    pred=model.predict(np.zeros((len(test_idx),1)))
                    ranked=pred[:,None]
                    margins=np.full(len(test_idx),np.nan)
                else:
                    model.fit(text[train_idx],y[train_idx])
                    pred=model.predict(text[test_idx])
                    ranked,margins=ranks_and_margin(model,text[test_idx])
                known=np.isin(y[test_idx],model.classes_)
                fold_stats.append({"fold":fold,"train_n":len(train_idx),"test_n":len(test_idx),
                    "test_groups_seen_in_training":len(shared_groups),"unseen_label_rows":int((~known).sum()),
                    "accuracy":float(accuracy_score(y[test_idx],pred)),"runtime_seconds":time.perf_counter()-started})
                for j,idx in enumerate(test_idx):
                    rows.append({"ID":complete.iloc[idx].ID,"fold":fold,"group":groups[idx],
                        "observed_code":y[idx],"predicted_code":pred[j],"top_2":ranked[j,1] if ranked.shape[1]>1 else "",
                        "top_3":ranked[j,2] if ranked.shape[1]>2 else "","top3_hit":bool(y[idx] in ranked[j]),
                        "label_seen_in_training":bool(known[j]),"correct":bool(y[idx]==pred[j]),
                        "margin_not_probability":float(margins[j])})
            result=pd.DataFrame(rows).sort_values("ID",key=lambda s:s.astype(int)).reset_index(drop=True)
            assert len(result)==len(complete) and result.ID.nunique()==len(complete)
            truth=result.observed_code.to_numpy()
            prediction=result.predicted_code.to_numpy()
            known=result.label_seen_in_training
            metrics={"model":kind,"fields":"core" if fields==CORE_FIELDS else "context_and_core", "split":split_kind,
                "n":len(result),"accuracy":float(accuracy_score(truth,prediction)),
                "macro_f1_all_59_classes":float(f1_score(truth,prediction,labels=classes,average="macro",zero_division=0)),
                "top3_accuracy":float(result.top3_hit.mean()) if kind!="dummy" else None,
                "first_digit_accuracy":float(np.mean([a[0]==b[0] for a,b in zip(truth,prediction)])),
                "unseen_label_rows":int((~known).sum()),
                "accuracy_seen_labels_only":float(result.loc[known,"correct"].mean()) if known.any() else None}
            return result,metrics,fold_stats

        experiments=[("dummy",TEXT_FIELDS,"grouped"), ("word",TEXT_FIELDS,"grouped"),
            ("char",TEXT_FIELDS,"grouped"), ("combined",TEXT_FIELDS,"grouped"),
            ("combined",CORE_FIELDS,"grouped"), ("combined",TEXT_FIELDS,"random_rows")]
        all_metrics=[]
        experiment_folds={}
        for kind,fields,split_kind in experiments:
            predictions,metrics,folds=evaluate(kind,fields,split_kind)
            all_metrics.append(metrics)
            key=kind+"_"+metrics["fields"]+"_"+split_kind
            experiment_folds[key]=folds
            if key=="combined_context_and_core_grouped":
                primary_predictions=predictions.copy()
                primary_metrics=metrics.copy()
        metrics_table=pd.DataFrame(all_metrics)
        print(metrics_table.to_string(index=False,float_format=lambda x:f"{x:.4f}"))
        print("Primary fold diagnostics:")
        print(pd.DataFrame(experiment_folds["combined_context_and_core_grouped"]).to_string(index=False))
        print("Random-row diagnostics, shown only to reveal overlap:")
        print(pd.DataFrame(experiment_folds["combined_context_and_core_random_rows"])[["fold","test_groups_seen_in_training","unseen_label_rows"]].to_string(index=False))


   model           fields       split   n  accuracy  macro_f1_all_59_classes  top3_accuracy  first_digit_accuracy  unseen_label_rows  accuracy_seen_labels_only
   dummy context_and_core     grouped 191    0.0419                   0.0026            NaN                0.3141                 84                     0.0748
    word context_and_core     grouped 191    0.3717                   0.2110         0.4084                0.5654                 84                     0.6636
    char context_and_core     grouped 191    0.3979                   0.2228         0.4869                0.5393                 84                     0.7103
combined context_and_core     grouped 191    0.3979                   0.2232         0.4712                0.5707                 84                     0.7103
combined             core     grouped 191    0.3717                   0.1899         0.4398                0.5288                 84                     0.6636
combined context_and_core random_rows 19

## 5. פירוש שגיאות ומגבלות
ציוני ההערכה מודדים התאמה לתוויות שסופקו, שחלקן עשויות להיות שגויות. אי אפשר להסיק מכך את שיעור הנכונות האמיתי. אין לתקן תווית רק מפני שהמודל חולק עליה. יש לבדוק את הרשומה מול כללי הסיווג ובקרת סמלן.

חלוקה אקראית של שורות מאפשרת לחלק מתיאורי התפקיד לחזור באימון ובבדיקה. ההשוואה שלה להפרדת קבוצות היא בדיקת רגישות, ולא אומדן בלתי תלוי של ביצועי ייצור. גם ההפרדה לפי תיאור זהה אינה מסירה את כל הדמיון בין רשומות.

מוצגים בהמשך כל ציוני המחלקות וכל השגיאות, ולא רק דוגמאות מוצלחות. המודל הראשי נשאר זה שנקבע מראש, גם אם גרסה אחרת מקבלת ציון גבוה יותר.


In [5]:
if df is None:
    print("Skipped: dataset not available.")
else:
    with report_failure("Assignment stage 5"):
        report=classification_report(primary_predictions.observed_code,primary_predictions.predicted_code,
            labels=classes,output_dict=True,zero_division=0)
        print(pd.DataFrame(report).transpose().to_string(float_format=lambda x:f"{x:.3f}"))
        errors=primary_predictions.loc[~primary_predictions.correct].copy()
        print("Errors:",len(errors),"of",len(primary_predictions))
        print(errors[["ID","observed_code","predicted_code","label_seen_in_training"]].to_string(index=False))
        # Equal weight per held-out task group is a sensitivity metric for repeated templates.
        group_accuracy=primary_predictions.groupby("group").correct.mean()
        print("Equal-group average accuracy:",round(float(group_accuracy.mean()),4))
        # Cluster bootstrap is descriptive split sensitivity, not a population confidence guarantee.
        rng=np.random.default_rng(SEED)
        boot=np.array([rng.choice(group_accuracy.to_numpy(),size=len(group_accuracy),replace=True).mean() for _ in range(2000)])
        print("Descriptive group bootstrap central 95% interval:",np.quantile(boot,[.025,.975]).round(4))


              precision  recall  f1-score  support
1212              0.000   0.000     0.000    1.000
1221              0.000   0.000     0.000    1.000
1330              0.000   0.000     0.000    1.000
1345              0.000   0.000     0.000    1.000
2113              0.000   0.000     0.000    1.000
2120              0.750   1.000     0.857    3.000
2131              0.750   1.000     0.857    3.000
2142              1.000   0.500     0.667    2.000
2262              0.000   0.000     0.000    2.000
2330              0.000   0.000     0.000    1.000
2342              0.714   1.000     0.833    5.000
2411              0.000   0.000     0.000    1.000
2412              0.000   0.000     0.000    1.000
2423              0.000   0.000     0.000    1.000
2431              0.000   0.000     0.000    1.000
2511              0.333   0.143     0.200    7.000
2512              0.556   1.000     0.714   10.000
2514              0.000   0.000     0.000    4.000
2631              1.000   0.333

## 6. הצעות לרשומות שסומלו חלקית
נאמן את המודל הראשי על 191 הרשומות עם קוד מלא. זהו מודל ההדגמה שמייצר הצעות לתשע הרשומות החלקיות. עבור קוד כמו 335X, מותרות רק הצעות ששומרות על הספרות הידועות. אם אף מחלקה באימון אינה תואמת, נשאיר את ההצעה ריקה. לכל הצעה נדרשת בדיקה אנושית. אין כאן שיעור דיוק, מפני שאין לרשומות האלה קוד אמת מלא.

ציון ההחלטה של LinearSVC אינו הסתברות. גם פער גדול בין ההצעות אינו אישור אוטומטי. בנתונים חסרים המודל עשוי להסתמך בעיקר על הקשר הארגון.


In [6]:
if df is None:
    print("Skipped: dataset not available.")
else:
    with report_failure("Assignment stage 6"):
        final_model=make_model("combined")
        final_model.fit(text_for(complete).to_numpy(),y)

        def compatible(code,pattern):
            return len(code)==len(pattern)==4 and all(p=="X" or p==c for c,p in zip(code,pattern))

        def suggest(frame,patterns=None):
            frame=frame.copy()
            for col in TEXT_FIELDS:
                if col not in frame:
                    frame[col]=""
            text=text_for(frame).to_numpy()
            scores=np.asarray(final_model.decision_function(text))
            if scores.ndim==1:
                scores=np.column_stack([-scores,scores])
            output=[]
            for pos,(_,row) in enumerate(frame.iterrows()):
                pattern=normalize(patterns[pos]).upper() if patterns is not None else "XXXX"
                if not re.fullmatch(r"[0-9X]{4}",pattern):
                    raise ValueError("Expected a four-character code pattern with digits or X.")
                allowed=[i for i,c in enumerate(final_model.classes_) if compatible(c,pattern)]
                ranked=sorted(allowed,key=lambda j:(-scores[pos,j],str(final_model.classes_[j])))[:3]
                candidates=[str(final_model.classes_[j]) for j in ranked]
                has_text=any(normalize(row[c]) for c in TEXT_FIELDS)
                if not has_text:
                    candidates=[]
                no_core=not any(normalize(row.get(c,"")) for c in CORE_FIELDS)
                reason="missing_all_text" if not has_text else "no_compatible_training_class" if not candidates else "missing_task_description" if no_core else "incomplete_reference_label"
                output.append({"ID":str(row.get("ID","")),"original_pattern":pattern,
                    "suggestion_1":candidates[0] if candidates else "",
                    "suggestion_2":candidates[1] if len(candidates)>1 else "",
                    "suggestion_3":candidates[2] if len(candidates)>2 else "",
                    "review_required":True,"review_reason":reason})
            return pd.DataFrame(output)

        partial_suggestions=suggest(partial,partial[TARGET].tolist())
        print(partial_suggestions.to_string(index=False))
        print("All suggestions require human review. No labels were changed.")


 ID original_pattern suggestion_1 suggestion_2 suggestion_3  review_required              review_reason
 18             1XXX         1345         1212         1221             True incomplete_reference_label
 45             XXXX         5131         5120         9412             True   missing_task_description
 50             1XXX         1345         1212         1221             True incomplete_reference_label
 81             1XXX         1345         1212         1221             True incomplete_reference_label
108             XXXX         5131         5120         9412             True   missing_task_description
113             1XXX         1345         1212         1221             True incomplete_reference_label
142             XXXX         5230         8183         5153             True   missing_task_description
165             335X         3354                                       True incomplete_reference_label
193             XXXX         3354         3512         7126     

## 7. בדיקות תקינות והפקת תוצרים
הבדיקות בוחנות קלט ריק, מילים שלא הופיעו באימון, שמירת ספרות ידועות בקוד חלקי, אפסים מובילים במחרוזת קוד, חוסר במחלקה תואמת והפרדה בין קבוצות. הן אינן מוכיחות התאמה לנתוני אמת או נכונות של הסימול המקורי.


In [7]:
if df is None:
    print("Skipped: dataset not available.")
else:
    with report_failure("Assignment stage 7"):
        checks={}
        checks["original_200_records"] = len(df)==200
        checks["label_partition"] = len(complete)==191 and len(partial)==9
        checks["complete_classes_59"] = len(classes)==59
        checks["exactly_one_heldout_prediction_per_record"] = len(primary_predictions)==191 and primary_predictions.ID.nunique()==191
        checks["no_primary_group_overlap"] = all(f["test_groups_seen_in_training"]==0 for f in experiment_folds["combined_context_and_core_grouped"])
        checks["no_target_or_postcoding_features"] = TARGET not in TEXT_FIELDS and "SemelAnafSofi" not in TEXT_FIELDS and "ID" not in TEXT_FIELDS
        checks["leading_zero_preserved"] = normalize("0110")=="0110" and compatible("0110","0XXX")
        checks["partial_pattern_constraints"] = compatible("3354","335X") and not compatible("3311","335X")
        blank=suggest(pd.DataFrame([{c:"" for c in TEXT_FIELDS}]))
        checks["empty_text_abstains"] = blank.suggestion_1.eq("").all() and blank.review_required.all()
        unknown=suggest(pd.DataFrame([{"EzoAvoda":"zzqxv newtokenלאמוכר"}]))
        checks["unknown_words_run_and_require_review"] = len(unknown)==1 and unknown.review_required.all()
        none_allowed=suggest(pd.DataFrame([{"EzoAvoda":"מתכנת"}]),["0000"])
        checks["no_compatible_class_abstains"] = none_allowed.suggestion_1.eq("").all()
        checks["prefixes_preserved_in_outputs"] = all(compatible(row[c],row.original_pattern) for _,row in partial_suggestions.iterrows() for c in ["suggestion_1","suggestion_2","suggestion_3"] if row[c])
        checks["all_partial_rows_require_review"] = bool(partial_suggestions.review_required.all())
        try:
            suggest(pd.DataFrame([{"EzoAvoda":"מתכנת"}]),["invalid"])
            checks["invalid_pattern_rejected"] = False
        except ValueError:
            checks["invalid_pattern_rejected"] = True
        checks={k:bool(v) for k,v in checks.items()}
        if not all(checks.values()):
            raise AssertionError({k:v for k,v in checks.items() if not v})
        print(json.dumps(checks,indent=2))

        primary_predictions.to_csv(OUTPUT/"heldout_predictions.csv",index=False,encoding="utf-8-sig")
        partial_suggestions.to_csv(OUTPUT/"partial_label_suggestions.csv",index=False,encoding="utf-8-sig")
        metrics_table.to_csv(OUTPUT/"model_comparison.csv",index=False,encoding="utf-8-sig")
        run={"dataset_sha256":DATA_SHA256,"versions":VERSIONS,"seed":SEED,"quality":quality,
             "primary_metrics":primary_metrics,"experiments":all_metrics,"folds":experiment_folds,
             "equal_group_accuracy":float(group_accuracy.mean()),
             "descriptive_group_bootstrap_interval":np.quantile(boot,[.025,.975]).tolist(),
             "checks":checks,"github_url":"https://github.com/shulruferor-sudo/Advance-Data-Science/blob/main/project_notebooks/cbs_occupation_classification/Assignment.ipynb","colab_execution_verified":False,
             "model_choice":"Combined text SVC fixed before evaluation. No tuning on held-out predictions.",
             "limitations":["Simulated sample", "No respondent identifier or submission timestamp", "No adjudicated gold labels",
                            "Near duplicates may span groups", "Closed class set", "No independent final test set", "No Colab runtime execution yet"]}
        (OUTPUT/"run_results.json").write_text(json.dumps(run,ensure_ascii=False,indent=2),encoding="utf-8")
        print("Results saved in",OUTPUT.resolve())
        print("Source checksum unchanged:",hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()==DATA_SHA256)


{
  "original_200_records": true,
  "label_partition": true,
  "complete_classes_59": true,
  "exactly_one_heldout_prediction_per_record": true,
  "no_primary_group_overlap": true,
  "no_target_or_postcoding_features": true,
  "leading_zero_preserved": true,
  "partial_pattern_constraints": true,
  "empty_text_abstains": true,
  "unknown_words_run_and_require_review": true,
  "no_compatible_class_abstains": true,
  "prefixes_preserved_in_outputs": true,
  "all_partial_rows_require_review": true,
  "invalid_pattern_rejected": true
}
Results saved in /workspace/scratch/98c53c9539cb/assignment-repo/project_notebooks/cbs_occupation_classification/results
Source checksum unchanged: True


## 8. מסקנות והמשך הטמעה
הפתרון מדגים שרשרת עבודה שניתנת לבדיקה, עם חיזויים מחוץ לאימון והפרדה בין תוויות מלאות להצעות לרשומות חלקיות. המדגם קטן ומדומה, והכיסוי של מחלקות רבות מצומצם. לכן אין הצדקה לאישור סימול אוטומטי על סמך התוצאות כאן.

להמשך נדרש מדגם מייצג עם מזהה נסקר וחותמת זמן לכל גרסת שאלון. יש ליצור מדגם אמת שהוכרע בידי סמלנים, להגדיר את גרסת מילון הסיווג ולבדוק מה זמין ברגע החיזוי. לאחר מכן אפשר להשוות את קו הבסיס למודל שפה עברי ולשליפת מועמדים ממילון מוסמך, באותה חלוקת נתונים ובאותם מדדים. אלה הצעות למחקר נוסף, ולא שיטות שנבדקו במחברת.

בשלב ראשון מומלץ ניסוי סיוע לסמלן: הצגת כמה מועמדים, תיעוד הבחירה והנימוק ובדיקה של טעויות ושל זמן טיפול. רק לאחר הערכה על נתוני אמת ובדיקת כיול אפשר לשקול סף לאישור אוטומטי. יש למדוד איכות גם במחלקות נדירות ובשדות חסרים, ולעקוב אחר שינוי בתיאורי עבודה ובמילון הסיווג.

## 9. מקורות והיקף השימוש
הדרישות נלקחו מהמסמך המקורי ומגיליונות הקובץ. ספריות התוכנה תומכות באופן המימוש, אך אינן מוכיחות את ביצועיו על הנתונים.

* ILO, ISCO-08, הגדרת עיסוק ומבנה היררכי: https://isco.ilo.org/en/isco-08/
* למ״ס, מילוני סיווג: https://www.cbs.gov.il/he/Pages/Code-list.aspx
* למ״ס, הודעה מ־30.4.2026 על טיוטת סיווג עיסוקים 2026: https://www.cbs.gov.il/he/mediarelease/DocLib/2026/132/20_26_132b.pdf
* scikit-learn, מניעת דליפת מידע: https://scikit-learn.org/stable/common_pitfalls.html
* scikit-learn, GroupKFold: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html
* scikit-learn, ייצוגי טקסט: https://scikit-learn.org/stable/modules/feature_extraction.html
* scikit-learn, F1: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html
* scikit-learn, LinearSVC: https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html
* Google Colab, הפעלת מחברות: https://research.google.com/colaboratory/faq.html

המקורות נבדקו ב־30.10.2026. ה־Metadata מגדיר קוד בן ארבע ספרות, ולכן זו הגדרת המטרה במחברת. פרסום על סיווג חדש אינו היתר להמיר את תוויות המטלה. אין כאן אימות של כל קוד מול מילון רשמי, ואין טענה שהנתונים זהים ל־ISCO הבינלאומי ללא ההתאמות בישראל.


## גרפים לעיון בתוצאות השמורות

אלו גרפים של ההרצה השמורה, ולא תחליף לחישוב מחדש לאחר שינוי נתונים או קוד. קוד ההפקה נמצא בסעיף CBS ב־Figures.ipynb.

![model_comparison](results/figures/model_comparison.png)

![split_sensitivity](results/figures/split_sensitivity.png)

![class_support](results/figures/class_support.png)

![missing_text](results/figures/missing_text.png)

